<a href="https://colab.research.google.com/github/samikshyasanskruti/Deep-Learning/blob/main/NN_with_Regularization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1.Import Libraries**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score
#from sklearn.linear_model import LogisticRegression

import warnings
warnings.filterwarnings('ignore')

**2.Preprocess the Dataset**

In [ ]:
# Get the data
url='https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
data=pd.read_csv(url,delimiter=';')


In [ ]:
data.head()

In [ ]:
data.describe()

In [ ]:
data.info()

 **3.Perform EDA**

In [ ]:
X=data.drop('quality',axis=1)
y=data['quality']

print(X.shape)
print(y.shape)

In [ ]:
print(y)

In [ ]:
#One-hot encoding

OHE=OneHotEncoder(sparse_output=False)
y_encoded=OHE.fit_transform(y.values.reshape(-1,1))
print(y_encoded)

In [ ]:
# split training and test split
X_train,X_test,y_train,y_test=train_test_split(X,y_encoded,test_size=0.2,random_state=42)

In [ ]:
#scaling

scaler=StandardScaler()
#scaler.fit(X_train)
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

**4.Build Model**

In [ ]:
# create the MLP model

import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input,Dense
from tensorflow.keras.regularizers import l1,l2
model=Sequential()
model.add(Input(shape=(X_train.shape[1],)))

model.add(Dense(32,activation='relu'))
model.add(Dense(64,activation='relu'))
model.add(Dense(128,activation='relu'))
model.add(Dense(y_train.shape[1],activation='softmax'))

model.summary()

**5.Train Model with Regularizer**

In [ ]:
#from tensorflow.keras.callbacks import EarlyStopping

early_stopping=tensorflow.keras.callbacks.EarlyStopping(monitor='val_loss',
                                                        patience=5,
                                                        restore_best_weights=True)
optimizer = tensorflow.keras.optimizers.Adam()
model.compile(optimizer=optimizer,loss='categorical_crossentropy',metrics=['accuracy'])
history=model.fit(X_train_scaled,y_train,
                  epochs=50,
                  validation_split=0.2,
                  callbacks=[early_stopping],
                  verbose=1)

**6.Plot Loss and Accuracy Curve**

In [ ]:
#printing training an validation loss/accuracy
plt.figure(figsize=(16,4))
plt.subplot(1,2,1)
plt.plot(history.history['loss'],label='Training Loss')
plt.plot(history.history['val_loss'],label='Validation Loss')
plt.legend()
plt.grid()

plt.subplot(1,2,2)
plt.plot(history.history['accuracy'],label='Training Accuracy')
plt.plot(history.history['val_accuracy'],label='Validation Accuracy')
plt.legend()
plt.grid()


**7.Classification Report**

In [ ]:
#prediction
y_pred=model.predict(X_test).round()


y_pred=np.argmax(y_pred, axis= 1)
y_test=np.argmax(y_test, axis= 1)

print('Accuracy:', accuracy_score(y_pred,y_test))
print(classification_report(y_pred,y_test))

**8.Confusion Matrix**

In [ ]:
cm=confusion_matrix(y_test,y_pred)

print(cm)
plt.figure(figsize=(5,3))
sns.heatmap(cm,annot=True,fmt='d')
plt.xlabel('predicted')
plt.ylabel('actual')

plt.show()